# Huấn luyện Custom DQN (Scratch) trên Atari Pong

Sổ tay này hướng dẫn huấn luyện mô hình **DQN (Deep Q-Network)** tự xây dựng từ đầu (scratch) bằng PyTorch để chơi game **Atari Pong** (môi trường `PongNoFrameskip-v4` của Gymnasium).

## Giới thiệu thuật toán và kiến trúc mô hình

### 1. Thuật toán DQN
DQN (Deep Q-Network) là một thuật toán học tăng cường dựa trên giá trị (Value-based Reinforcement Learning). Nó kết hợp Q-Learning cổ điển với Mạng Nơ-ron Nhân tạo Sâu (Deep Neural Networks) để xấp xỉ hàm giá trị hành động $Q(s, a)$.

Các thành phần quan trọng trong DQN bao gồm:
- **Bộ đệm tái hiện trải nghiệm (Replay Buffer)**: Lưu trữ các bộ dữ liệu chuyển trạng thái $(s, a, r, s', done)$ để lấy mẫu ngẫu nhiên dạng mini-batch, giúp loại bỏ tính tương quan tuyến tính giữa các mẫu liên tiếp và làm ổn định quá trình tối ưu hóa gradient.
- **Mạng mục tiêu (Target Network)**: Một bản sao độc lập của mạng hành vi (Online Network), được cập nhật chậm hơn nhằm giữ cho các giá trị mục tiêu (target $y$) không bị dao động liên tục khi tính toán hàm mất mát Bellman.

### 2. Kiến trúc mô hình mạng tích chập (CNN)
Mô hình sử dụng kiến trúc Nature CNN để xử lý các frame ảnh trực tiếp từ game Pong:
- Đầu vào: Tập hợp 4 khung ảnh grayscale xếp chồng liền kề có kích thước $4 	imes 84 	imes 84$ để nắm bắt thông tin chuyển động của quả bóng.
- Các lớp tích chập (Convolutional Layers):
  - Lớp 1: `Conv2d(in_channels=4, out_channels=32, kernel_size=8, stride=4)` + `ReLU`
  - Lớp 2: `Conv2d(in_channels=32, out_channels=64, kernel_size=4, stride=2)` + `ReLU`
  - Lớp 3: `Conv2d(in_channels=64, out_channels=64, kernel_size=3, stride=1)` + `ReLU`
- Các lớp kết nối đầy đủ (Fully Connected Layers):
  - Lớp ẩn: `Linear(feature_size, 512)` + `ReLU`
  - Lớp đầu ra: `Linear(512, num_actions)` dự đoán Q-values cho từng hành động hợp lệ.

## Cấu hình môi trường và Import các thư viện

Để chạy notebook này độc lập trong thư mục con `notebooks/`, chúng ta cần thêm thư mục gốc của dự án vào đường dẫn tìm kiếm `sys.path` để import chính xác các module từ `src/`.

In [ ]:
import sys
import os
import time
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt

# Thêm thư mục gốc vào đường dẫn hệ thống để import src
sys.path.append(os.path.abspath(os.path.join('..')))

from src.common.wrappers import make_atari_env
from src.common.utils import CSVLogger, save_checkpoint
from src.scratch.agent import DQNAgent

## Kiểm tra tăng tốc phần cứng (GPU CUDA)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Thiết bị huấn luyện: {device}")
if torch.cuda.is_available():
    print(f"Tên GPU: {torch.cuda.get_device_name(0)}")

## Định nghĩa Siêu tham số (Hyperparameters)

Cấu hình các siêu tham số huấn luyện trực tiếp ngay trong notebook:

In [ ]:
env_id = "PongNoFrameskip-v4"
total_steps = 1000000       # Tổng số bước môi trường huấn luyện
lr = 1e-4                  # Tốc độ học của optimizer Adam
buffer_size = 100000       # Dung lượng tối đa của Replay Buffer
batch_size = 32            # Kích thước tập mẫu huấn luyện
target_update_freq = 10000 # Chu kỳ cập nhật trọng số cho Target Network
learning_starts = 10000    # Số bước thu thập dữ liệu khởi tạo trước khi bắt đầu cập nhật mạng
train_freq = 4             # Tần suất tối ưu hóa mạng (sau mỗi 4 bước môi trường)
seed = 42                  # Random seed để tái hiện kết quả

# Thư mục lưu kết quả học tập và các file mô hình
save_dir = "../data/models"
log_dir = "../data/logs"

os.makedirs(save_dir, exist_ok=True)
os.makedirs(log_dir, exist_ok=True)

## Khởi tạo Môi trường Pong với Wrappers thống nhất

Sử dụng hàm `make_atari_env` để khởi tạo môi trường Pong cùng các wrapper tiêu chuẩn (Noop reset, Frame Skip, Grayscale, Resize 84x84, Normalization, Frame Stack 4).

In [ ]:
env = make_atari_env(env_id)
env.action_space.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)

print(f"Trạng thái đầu vào: {env.observation_space.shape}")
print(f"Số lượng hành động: {env.action_space.n}")

## Khởi tạo Tác nhân (DQN Agent)

In [ ]:
agent = DQNAgent(
    state_shape=env.observation_space.shape,
    num_actions=env.action_space.n,
    learning_rate=lr,
    gamma=0.99,
    buffer_size=buffer_size,
    batch_size=batch_size,
    target_update_freq=target_update_freq,
    device=device,
    dueling=False,
    double=False  # Standard DQN không dùng Double update
)

## Vòng lặp Huấn luyện (Training Loop)

Vòng lặp chạy huấn luyện hoàn chỉnh, ghi log định kỳ vào file CSV và tự động lưu checkpoint mô hình cứ sau mỗi 100,000 steps.

In [ ]:
run_name = f"dqn_scratch_seed{seed}_{int(time.time())}"
csv_file = os.path.join(log_dir, f"{run_name}.csv")
csv_logger = CSVLogger(
    filepath=csv_file,
    headers=["step", "episode", "reward", "epsilon", "loss", "duration"]
)

state, _ = env.reset(seed=seed)
episode_reward = 0.0
episode_steps = 0
episode_count = 0
episode_start_time = time.time()
episode_losses = []
best_mean_reward = -float("inf")

epsilon_start = 1.0
epsilon_end = 0.01
exploration_fraction = 0.1
epsilon_decay_steps = int(total_steps * exploration_fraction)

print(f"Bắt đầu huấn luyện Standard DQN Scratch trên {env_id}...")

for step in range(1, total_steps + 1):
    # Suy giảm epsilon tuyến tính
    epsilon = max(
        epsilon_end,
        epsilon_start - (epsilon_start - epsilon_end) * (step / epsilon_decay_steps)
    )
    
    # Chọn hành động epsilon-greedy
    action = agent.select_action(state, epsilon)
    
    # Thực hiện hành động trong môi trường
    next_state, reward, terminated, truncated, _ = env.step(action)
    done = terminated or truncated
    
    # Lưu trải nghiệm vào Replay Buffer
    agent.replay_buffer.add(state, action, reward, next_state, done)
    
    state = next_state
    episode_reward += reward
    episode_steps += 1
    
    # Tối ưu hóa mạng online_net
    if step > learning_starts and step % train_freq == 0:
        loss_val = agent.update()
        episode_losses.append(loss_val)
        
    # Cập nhật Target Network
    if step % target_update_freq == 0:
        agent.update_target_network()
        print(f"Bước {step}/{total_steps} | Đồng bộ trọng số Target Network.")
        
    # Kết thúc một episode
    if done:
        episode_duration = time.time() - episode_start_time
        episode_count += 1
        avg_loss = np.mean(episode_losses) if len(episode_losses) > 0 else 0.0
        
        # Log số liệu tập phim
        csv_logger.log({
            "step": step,
            "episode": episode_count,
            "reward": episode_reward,
            "epsilon": epsilon,
            "loss": avg_loss,
            "duration": episode_duration
        })
        
        if episode_count % 10 == 0 or episode_reward > -10:
            print(
                f"Tập {episode_count:4d} | Bước {step:6d} | "
                f"Phần thưởng: {episode_reward:5.1f} | Epsilon: {epsilon:.3f} | "
                f"Loss Trung Bình: {avg_loss:.5f} | Thời gian: {episode_duration:.1f}s"
            )
            
        # Lưu mô hình tốt nhất
        if episode_reward > best_mean_reward and step > learning_starts:
            best_mean_reward = episode_reward
            save_checkpoint(
                state={
                    "step": step,
                    "episode": episode_count,
                    "online_net_state_dict": agent.online_net.state_dict(),
                    "reward": episode_reward,
                    "epsilon": epsilon
                },
                filepath=os.path.join(save_dir, "dqn_scratch_best.pt")
            )
            
        # Khởi động lại các tham số tập phim mới
        state, _ = env.reset()
        episode_reward = 0.0
        episode_steps = 0
        episode_start_time = time.time()
        episode_losses = []
        
    # Lưu checkpoint định kỳ mỗi 100,000 steps
    if step % 100000 == 0:
        save_checkpoint(
            state={
                "step": step,
                "episode": episode_count,
                "online_net_state_dict": agent.online_net.state_dict(),
                "reward": episode_reward
            },
            filepath=os.path.join(save_dir, f"dqn_scratch_step_{step}.pt")
        )

env.close()
print("Quá trình huấn luyện kết thúc thành công!")

## Vẽ đồ thị kết quả huấn luyện

Đọc file log CSV vừa tạo và trực quan hóa đường cong episodic reward cùng loss bằng Matplotlib.

In [ ]:
try:
    df = pd.read_csv(csv_file)
    
    # Tạo hình vẽ
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))
    
    # 1. Đồ thị episodic reward
    ax1.plot(df["episode"], df["reward"], alpha=0.2, color="blue", label="Thực tế")
    rolling_rew = df["reward"].rolling(window=20, min_periods=1).mean()
    ax1.plot(df["episode"], rolling_rew, color="red", linewidth=2, label="Trung bình trượt (20 tập)")
    ax1.set_title("Đường cong học tập - Phần thưởng tập phim")
    ax1.set_xlabel("Tập phim (Episode)")
    ax1.set_ylabel("Phần thưởng (Reward)")
    ax1.legend()
    
    # 2. Đồ thị loss trung bình
    valid_loss_df = df[df["loss"] > 0]
    ax2.plot(valid_loss_df["episode"], valid_loss_df["loss"], color="purple", alpha=0.5, label="Hàm mất mát")
    rolling_loss = valid_loss_df["loss"].rolling(window=20, min_periods=1).mean()
    ax2.plot(valid_loss_df["episode"], rolling_loss, color="darkorange", linewidth=2, label="Trượt trượt Loss")
    ax2.set_title("Lịch sử mất mát của mô hình (TD-Loss)")
    ax2.set_xlabel("Tập phim (Episode)")
    ax2.set_ylabel("Loss")
    ax2.legend()
    
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"Không thể vẽ đồ thị: {e}. Hãy kiểm tra xem file CSV đã được ghi nhận đầy đủ số liệu chưa.")